# Level 2 autonomous vision starter

이 notebook은 Google Colab, Google Drive, Jupyter, VS Code에서 standalone으로 실행할 수 있는 starter입니다.
아래 setup cell을 먼저 실행하면 GitHub의 최신 `menlo_runner` package를 설치합니다.


## Project 규칙

Level 2에서는 scene_state, 정확한 entity ID, coordinate go_to를 사용할 수 없습니다. Camera, set_head, set_velocity, memory로 navigation을 구현하세요.

Code cell의 `학생 TODO` 부분을 팀 전략에 맞게 구현하세요. 기본 실행 cell은 10분 scored simulation을 실행합니다.


In [ ]:
# Colab/로컬 setup입니다. 필요하면 한 번 실행하세요.
%pip install -q "git+https://github.com/menlo-ai/menlo-runner.git"

# 로컬 clone repo에서 실행 중이면 위 install cell을 건너뛰어도 됩니다.


## Starter code

아래 code cell에는 Python starter와 같은 runnable code가 들어 있습니다. 설명, TODO, comment는 한국어로 작성되어 있습니다.


In [ ]:
from __future__ import annotations

"""Menlo AI 로봇 분류 챌린지용 Level 2 프로젝트 시작 파일입니다.

이 파일은 완성된 해답이 아니라 시작 파일입니다.

지원 코드 섹션은 반복해서 작성할 필요가 없는 작은 래퍼와 자료 구조를 제공합니다.
필요하면 읽고 수정할 수 있지만, 대부분의 팀은 지원 코드를 크게 바꾸지 않는 편이 좋습니다.
학생 TODO 섹션은 팀이 수정하고, 개선하고, test하고, presentation에서 설명해야 하는 부분입니다.

Level 2 규칙: scene_state, 정확한 entity ID, coordinate go_to는 사용할 수 없습니다.
Camera observation, set_head, set_velocity, memory로 navigation을 구현하세요.
"""

import asyncio
import json
import math
import os
from dataclasses import dataclass, field
from typing import Any

from menlo_runner.completion import CompletionConfig, CompletionTracker
from menlo_runner.llm import ask_vlm
from menlo_runner.perception import detect_color_blobs


# ---------------------------------------------------------------------------
# 지원 코드: 공통 과제 정의와 필수 LLM 결정 형식
# ---------------------------------------------------------------------------
# 과제 문장은 고정합니다. 목표는 cube 색상 순서와 시작 위치가 달라져도
# 소스 코드 변경 없이 처리하는 하나의 agent를 만드는 것입니다.
TASK = "Find and sort cubes from the source area into their matching destination pads."

# 고정 표지판 정보는 사용할 수 있습니다. 단, 이를 정확한 coordinate나 entity ID로
# 바꾸지 말고 관찰을 해석하는 데만 사용하세요.
DESTINATION_SIGN_RULES = {
    "red": "B",
    "green": "C",
    "blue": "D",
    "yellow": "E",
}
SIGNAGE_NOTE = (
    "A는 conveyor/cube source area이며 destination이 아닙니다. "
    "Destination sign은 B red, C green, D blue, E yellow입니다."
)

# LLM은 아래 set에서 상위 단계 행동을 선택해야 합니다. 원시 속도 명령을
# 직접 출력하지 말고, 결정적 코드가 결정을 robot 행동으로 변환해야 합니다.
ALLOWED_NEXT_ACTIONS = {
    "search_cube",
    "navigate_to_cube",
    "pick_cube",
    "search_pad",
    "navigate_to_pad",
    "place_cube",
    "recover",
    "skip_target",
    "stop",
}


@dataclass
class AgentDecision:
    """LLM이 반환하고 코드가 검증한 상위 단계 결정입니다."""

    next_action: str
    target_color: str | None = None
    reason: str = ""
    recovery_strategy: str | None = None


@dataclass
class AgentMemory:
    """observe-decide-act cycle 사이에 agent가 유지하는 상태입니다.

    간단하게 시작한 뒤, 팀 전략에 필요한 field를 추가하세요. 예: target history,
    failed location, scan result, confidence score, held-object estimate 등.
    """

    delivered_count: int = 0
    delivery_limit: int | None = None
    priority_colors: list[str] = field(default_factory=list)
    held_color: str | None = None
    active_color: str | None = None
    stage: str = "need_cube"
    search_turns: int = 0
    failed_attempts: dict[str, int] = field(default_factory=dict)
    completed_colors: list[str] = field(default_factory=list)
    skipped_colors: list[str] = field(default_factory=list)
    logs: list[dict[str, Any]] = field(default_factory=list)


@dataclass
class Observation:
    """LLM과 실행 코드에 전달할 간결한 관찰입니다."""

    robot_status: Any
    detections: list[Any]
    note: str = ""
    vlm_summary: str = ""


@dataclass(frozen=True)
class ScannedDetection:
    """해당 camera frame을 얻을 때 사용한 head pose가 함께 기록된 color detection입니다.

    이 구조는 특정 strategy에 묶이지 않도록 의도적으로 중립적입니다. Level 1 팀은 coordinate estimate에 full bearing을 사용할 수 있고, Level 2 팀은 closed-loop visual centering에 사용할 수 있습니다. 필요하면 confidence, target type, depth field를 추가하세요.
    """

    color: str
    angle_deg: float
    blob_area: int
    centroid: tuple[int, int]
    bbox: tuple[int, int, int, int]
    head_yaw: float
    head_pitch: float

    @property
    def full_bearing_deg(self) -> float:
        """대략적인 body-relative bearing입니다. Image angle에 head yaw를 더합니다."""
        return self.angle_deg + math.degrees(self.head_yaw)


def parse_agent_decision(text: str) -> AgentDecision | None:
    """필수 structured LLM JSON output을 parse하고 validate합니다."""
    stripped = text.strip()
    if stripped.startswith("```"):
        stripped = stripped.strip("`")
        if stripped.lower().startswith("json"):
            stripped = stripped[4:].strip()
    if not stripped.startswith("{"):
        start = stripped.find("{")
        end = stripped.rfind("}")
        if start >= 0 and end > start:
            stripped = stripped[start : end + 1]
    try:
        data = json.loads(stripped)
    except json.JSONDecodeError:
        return None

    next_action = data.get("next_action")
    if next_action not in ALLOWED_NEXT_ACTIONS:
        return None

    target_color = data.get("target_color")
    if target_color is not None and not isinstance(target_color, str):
        return None

    return AgentDecision(
        next_action=next_action,
        target_color=target_color,
        reason=str(data.get("reason", "")),
        recovery_strategy=data.get("recovery_strategy"),
    )


def build_decision_context(
    task: str,
    observation: Observation,
    memory: AgentMemory,
    last_result: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """Robot state를 LLM에 전달하기 좋은 간결한 text context로 변환합니다.

    VLM을 명시적으로 사용하는 경우가 아니라면 raw image는 이 text context에 넣지 마세요. LLM은 다음 high-level step을 고를 만큼의 정보만 받고, low-level control과 safety는 code가 처리해야 합니다.
    """
    visible = [
        {
            "color": detection.color,
            "angle_deg": detection.angle_deg,
            "full_bearing_deg": round(getattr(detection, "full_bearing_deg", detection.angle_deg), 1),
            "blob_area": detection.blob_area,
            "bbox": detection.bbox,
        }
        for detection in observation.detections
    ]
    return {
        "task": task,
        "visible_targets": visible,
        "held_color": memory.held_color,
        "active_color": memory.active_color,
        "stage": memory.stage,
        "delivered_count": memory.delivered_count,
        "completed_colors": memory.completed_colors,
        "skipped_colors": memory.skipped_colors,
        "failed_attempts": memory.failed_attempts,
        "last_result": last_result,
        "note": observation.note,
        "signage_note": SIGNAGE_NOTE,
        "vlm_summary": observation.vlm_summary,
    }


# ---------------------------------------------------------------------------
# 지원 코드: project 규칙에 맞는 SDK wrapper
# ---------------------------------------------------------------------------
# 이 래퍼들은 프로젝트 규칙에 맞는 input을 노출합니다.
# Ground-truth coordinate, 정확한 target ID, global asset map은 추가하지 마세요.

async def get_robot_status(ctx: Any) -> Any:
    """Robot pose, motion status, neck state를 읽습니다."""
    return await ctx.state("robot_status")


async def get_camera_frame(ctx: Any) -> bytes:
    """POV camera frame을 가져옵니다."""
    return await ctx.get_vision("pov")


def build_signage_vlm_prompt(held_color: str | None = None) -> str:
    """고정 warehouse signage를 읽기 위한 strategy-neutral prompt를 만듭니다."""
    target = ""
    if held_color in DESTINATION_SIGN_RULES:
        target = f" Robot이 {held_color} cube를 들고 있으므로 target destination sign은 {DESTINATION_SIGN_RULES[held_color]}입니다."
    return (
        "이 robot camera frame에 보이는 warehouse sign을 읽으세요. "
        f"{SIGNAGE_NOTE} "
        "보이는 sign letter, color, 대략적인 left/center/right 위치, confidence를 JSON으로 반환하세요."
        + target
    )


async def ask_vlm_about_frame(ctx: Any, prompt: str, *, api_key: str) -> str:
    """Project에서 허용되는 VLM helper로 현재 POV frame에 대해 질문합니다."""
    jpeg = await get_camera_frame(ctx)
    return ask_vlm(jpeg, prompt, api_key=api_key)


async def perceive(ctx: Any) -> list[Any]:
    """현재 camera frame에서 Workshop 2 color-blob detector를 실행합니다."""
    jpeg = await get_camera_frame(ctx)
    return detect_color_blobs(jpeg)


async def set_head(ctx: Any, *, yaw: float | None = None, pitch: float | None = None) -> Any:
    """Walking direction을 바꾸지 않고 camera 방향을 조정합니다."""
    args: dict[str, float] = {}
    if yaw is not None:
        args["yaw"] = yaw
    if pitch is not None:
        args["pitch"] = pitch
    return await ctx.invoke("set_head", args, timeout_s=10)


async def move_velocity(
    ctx: Any,
    *,
    vx: float = 0.0,
    vy: float = 0.0,
    wz: float = 0.0,
    duration_s: float = 1.0,
) -> Any:
    """짧은 body-frame velocity command를 보낸 뒤 멈춥니다."""
    return await ctx.invoke(
        "set_velocity",
        {"vx": vx, "vy": vy, "wz": wz, "duration_s": duration_s},
        timeout_s=30,
    )


async def cancel_action(ctx: Any) -> Any:
    """현재 실행 중인 runtime action을 취소합니다."""
    return await ctx.invoke("cancel", {})


async def pick_nearest_cube(ctx: Any) -> Any:
    """Code가 robot을 시각적으로 충분히 위치시킨 뒤 nearest cube를 집습니다."""
    return await ctx.invoke(
        "pick_entity",
        {"target": {"kind": "entity", "entity_id": "cube"}},
        timeout_s=300,
    )


async def place_nearest_zone(ctx: Any) -> Any:
    """Matching pad에 도달한 뒤 nearest zone에 place합니다."""
    return await ctx.invoke("place_entity", {}, timeout_s=300)


def result_summary(result: Any) -> dict[str, Any]:
    """SDK result를 log하기 쉬운 작은 dictionary로 변환합니다."""
    error = getattr(result, "error", None)
    status = getattr(result, "status", None)
    return {
        "status": str(status) if status is not None else None,
        "error": getattr(error, "message", None) if error else None,
    }


async def scan_head(
    ctx: Any,
    *,
    yaws: tuple[float, ...] = (-0.8, 0.0, 0.8),
    pitch: float = 0.15,
) -> list[Any]:
    """간단한 scan helper입니다. 더 나은 search 전략으로 교체할 수 있습니다."""
    all_detections: list[Any] = []
    for yaw in yaws:
        await set_head(ctx, yaw=yaw, pitch=pitch)
        await asyncio.sleep(0.4)
        for detection in await perceive(ctx):
            all_detections.append(
                ScannedDetection(
                    color=detection.color,
                    angle_deg=detection.angle_deg,
                    blob_area=detection.blob_area,
                    centroid=detection.centroid,
                    bbox=detection.bbox,
                    head_yaw=yaw,
                    head_pitch=pitch,
                )
            )
    return all_detections


# ---------------------------------------------------------------------------
# 학생 TODO: LLM decision 함수
# ---------------------------------------------------------------------------
async def decide_next_action(
    task: str,
    observation: Observation,
    memory: AgentMemory,
    last_result: dict[str, Any] | None = None,
) -> AgentDecision:
    """LLM으로 다음 상위 단계 행동을 선택하고 stage-aware fallback으로 방어합니다.

    LLM은 high-level JSON action만 선택합니다. 속도값, 좌표, entity id는
    deterministic execution code가 관리하며 LLM 출력은 parse_agent_decision으로
    검증합니다.
    """
    decision_context = build_decision_context(task, observation, memory, last_result)
    system_prompt = "\n".join(
        [
            "You are the high-level supervisor for a Level 2 vision-only warehouse robot.",
            "Return ONLY one JSON object, no markdown.",
            "Allowed next_action values: " + ", ".join(sorted(ALLOWED_NEXT_ACTIONS)) + ".",
            'Schema: {"next_action":"search_cube|navigate_to_cube|pick_cube|search_pad|navigate_to_pad|place_cube|recover|skip_target|stop", "target_color": null, "reason": "short reason", "recovery_strategy": null}',
            "Never output coordinates, entity ids, velocity values, or low-level robot commands.",
            "If held_color is set, work only on the matching destination pad for that color.",
            "If not holding a cube, work only on finding, navigating to, or picking a cube.",
            "Use recover/skip_target for repeated failures; stop only when the completion condition is reached or the task is impossible.",
        ]
    )

    def stage_safe(parsed: AgentDecision | None) -> AgentDecision | None:
        if parsed is None or parsed.next_action not in ALLOWED_NEXT_ACTIONS:
            return None
        if memory.delivery_limit is not None and memory.delivered_count >= memory.delivery_limit:
            return AgentDecision(next_action="stop", reason="Delivery limit reached.")

        if memory.held_color:
            if parsed.next_action in {"search_cube", "navigate_to_cube", "pick_cube"}:
                return None
            if parsed.target_color not in {None, memory.held_color}:
                return None
            parsed.target_color = memory.held_color
            if parsed.next_action == "place_cube" and memory.stage != "ready_to_place":
                return None
            return parsed

        if parsed.next_action in {"search_pad", "navigate_to_pad", "place_cube"}:
            return None
        if parsed.next_action == "pick_cube" and memory.stage != "ready_to_pick":
            return None
        return parsed

    parsed: AgentDecision | None = None
    api_key = os.environ.get("TOKAMAK_API_KEY", "")
    if api_key:
        try:
            from menlo_runner.llm import call_llm

            reply = await asyncio.to_thread(
                call_llm,
                [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": json.dumps(decision_context, ensure_ascii=False)},
                ],
                api_key=api_key,
            )
            parsed = stage_safe(parse_agent_decision(reply))
        except Exception as exc:  # LLM/network failures must not crash the robot loop.
            memory.logs.append({"llm_fallback": type(exc).__name__, "stage": memory.stage})

    if parsed is not None:
        return parsed

    visible = decision_context["visible_targets"]
    repeated_failures = max(memory.failed_attempts.values(), default=0)
    active_color = memory.active_color or (memory.priority_colors[0] if memory.priority_colors else None)

    if memory.delivery_limit is not None and memory.delivered_count >= memory.delivery_limit:
        return AgentDecision(next_action="stop", reason="Delivery limit reached.")
    if repeated_failures >= 5:
        return AgentDecision(next_action="recover", target_color=memory.held_color or active_color, reason="Repeated failures triggered recovery.", recovery_strategy="wide_backoff_rescan")

    if memory.held_color:
        target_color = memory.held_color
        matching_visible = [item for item in visible if item["color"] == target_color]
        if memory.stage == "ready_to_place":
            return AgentDecision(next_action="place_cube", target_color=target_color, reason="Matching pad proximity was visually confirmed.")
        if matching_visible:
            return AgentDecision(next_action="navigate_to_pad", target_color=target_color, reason="Matching destination color is visible while holding cube.")
        return AgentDecision(next_action="search_pad", target_color=target_color, reason="Holding cube; search for matching destination pad.")

    if memory.stage == "ready_to_pick" and memory.active_color:
        return AgentDecision(next_action="pick_cube", target_color=memory.active_color, reason="Cube proximity was visually confirmed.")

    if active_color:
        matching_visible = [item for item in visible if item["color"] == active_color]
        if matching_visible:
            return AgentDecision(next_action="navigate_to_cube", target_color=active_color, reason="Active target cube color is visible.")

    if visible:
        largest = max(visible, key=lambda item: item["blob_area"])
        return AgentDecision(next_action="navigate_to_cube", target_color=largest["color"], reason="Largest visible cube-color blob selected by safe fallback.")

    return AgentDecision(next_action="search_cube", target_color=active_color, reason="No target visible; search for a cube.")


# ---------------------------------------------------------------------------
# 학생 TODO: observation, execution, verification, memory
# ---------------------------------------------------------------------------
async def observe_world(ctx: Any, memory: AgentMemory) -> Observation:
    """현재 stage에 맞춰 camera/head 기반 observation을 수집합니다."""
    robot_status = await get_robot_status(ctx)
    wide_scan_stages = {"need_cube", "search_cube", "holding_cube", "search_pad", "recover"}
    if memory.stage in wide_scan_stages:
        detections = await scan_head(ctx, yaws=(-0.9, -0.45, 0.0, 0.45, 0.9), pitch=0.15)
        note = "wide_scan"
    else:
        await set_head(ctx, yaw=0.0, pitch=0.15)
        await asyncio.sleep(0.2)
        detections = [
            ScannedDetection(
                color=d.color,
                angle_deg=d.angle_deg,
                blob_area=d.blob_area,
                centroid=d.centroid,
                bbox=d.bbox,
                head_yaw=0.0,
                head_pitch=0.15,
            )
            for d in await perceive(ctx)
        ]
        note = "front_frame"
    return Observation(robot_status=robot_status, detections=detections, note=note)


async def verify_outcome(
    ctx: Any,
    decision: AgentDecision,
    action_result: dict[str, Any],
    memory: AgentMemory | None = None,
) -> dict[str, Any]:
    """SDK result와 직후 camera evidence를 구조화합니다.

    Level 2 agent logic은 scene_state를 읽지 않습니다. Held/delivered 상태는
    action result와 memory의 의도 색상에서 보수적으로 추정하고, official
    runtime scoring은 별도 completion/harness 책임으로 둡니다.
    """
    result_payload = action_result.get("result") if isinstance(action_result, dict) else None
    status = None
    error = None
    if isinstance(result_payload, dict):
        status = result_payload.get("status")
        error = result_payload.get("error")
    elif result_payload is not None:
        status = getattr(result_payload, "status", None)
        sdk_error = getattr(result_payload, "error", None)
        error = getattr(sdk_error, "message", None) if sdk_error else None
    elif isinstance(action_result, dict):
        status = action_result.get("status")
        error = action_result.get("error")

    detections: list[Any] = []
    target_seen = False
    max_target_area = 0
    if decision.next_action not in {"stop", "skip_target"}:
        try:
            detections = await scan_head(ctx, yaws=(-0.35, 0.0, 0.35), pitch=0.15)
            target_seen = any(d.color == decision.target_color for d in detections) if decision.target_color else bool(detections)
            max_target_area = max(
                (d.blob_area for d in detections if decision.target_color is None or d.color == decision.target_color),
                default=0,
            )
        except Exception as exc:
            return {
                "ok": False,
                "decision": decision.__dict__,
                "action_result": action_result,
                "delivered_count": memory.delivered_count if memory is not None else 0,
                "held_cube": None,
                "held_color": memory.held_color if memory is not None else None,
                "error": f"post_action_observation_failed:{type(exc).__name__}",
            }

    status_text = str(status or action_result.get("status", "")).lower() if isinstance(action_result, dict) else str(status or "").lower()
    sdk_ok = (not error) and any(token in status_text for token in ("done", "success", "succeed", "complete", "ok"))

    if decision.next_action in {"search_cube", "search_pad"}:
        ok = bool(action_result.get("found"))
    elif decision.next_action in {"navigate_to_cube", "navigate_to_pad"}:
        ok = bool(action_result.get("reached"))
    elif decision.next_action in {"pick_cube", "place_cube"}:
        action_status = str(action_result.get("status", ""))
        blocked_by_precondition = action_status.startswith("blocked_")
        ok = sdk_ok or (not blocked_by_precondition and not error and bool(result_payload))
    elif decision.next_action == "recover":
        ok = str(action_result.get("status", "")).startswith("recovered")
    elif decision.next_action in {"skip_target", "stop"}:
        ok = True
    else:
        ok = False

    delivered_count = memory.delivered_count if memory is not None else 0
    held_color = memory.held_color if memory is not None else None
    if ok and decision.next_action == "pick_cube":
        held_color = decision.target_color or (memory.active_color if memory is not None else None)
    elif ok and decision.next_action == "place_cube":
        delivered_count += 1
        held_color = None

    return {
        "ok": ok,
        "decision": decision.__dict__,
        "action_result": action_result,
        "delivered_count": delivered_count,
        "held_cube": None,
        "held_color": held_color,
        "sdk_status": str(status) if status is not None else None,
        "sdk_error": error,
        "post_visible_count": len(detections),
        "target_seen": target_seen,
        "max_target_area": max_target_area,
    }


def update_memory(
    memory: AgentMemory,
    observation: Observation,
    decision: AgentDecision,
    verified: dict[str, Any],
) -> None:
    """각 cycle 뒤 stage, held/active color, 실패 횟수, log를 갱신합니다."""
    ok = bool(verified.get("ok"))
    color = decision.target_color or memory.active_color or memory.held_color
    failure_key = f"{decision.next_action}:{color or 'none'}"

    if "delivered_count" in verified:
        memory.delivered_count = int(verified.get("delivered_count") or 0)

    if ok:
        memory.failed_attempts.pop(failure_key, None)
    elif decision.next_action != "stop":
        memory.failed_attempts[failure_key] = memory.failed_attempts.get(failure_key, 0) + 1

    if decision.next_action == "search_cube":
        memory.stage = "approaching_cube" if ok else "search_cube"
        memory.search_turns += 1
    elif decision.next_action == "navigate_to_cube":
        memory.active_color = decision.target_color or memory.active_color
        memory.stage = "ready_to_pick" if ok else "search_cube"
    elif decision.next_action == "pick_cube":
        if ok:
            picked_color = verified.get("held_color") or color
            memory.held_color = picked_color
            memory.active_color = picked_color
            memory.stage = "holding_cube"
            memory.search_turns = 0
        else:
            memory.stage = "search_cube"
    elif decision.next_action == "search_pad":
        memory.stage = "approaching_pad" if ok else "search_pad"
        memory.search_turns += 1
    elif decision.next_action == "navigate_to_pad":
        memory.stage = "ready_to_place" if ok else "search_pad"
    elif decision.next_action == "place_cube":
        if ok:
            delivered_color = memory.held_color or color
            if delivered_color and delivered_color not in memory.completed_colors:
                memory.completed_colors.append(delivered_color)
            memory.held_color = None
            memory.active_color = None
            memory.search_turns = 0
            if memory.delivery_limit is not None and memory.delivered_count >= memory.delivery_limit:
                memory.stage = "done"
            else:
                memory.stage = "need_cube"
        else:
            memory.stage = "search_pad"
    elif decision.next_action == "recover":
        memory.stage = "search_pad" if memory.held_color else "search_cube"
    elif decision.next_action == "skip_target":
        if color and color not in memory.skipped_colors:
            memory.skipped_colors.append(color)
        memory.active_color = None
        memory.stage = "holding_cube" if memory.held_color else "need_cube"
    elif decision.next_action == "stop":
        memory.stage = "done"

    memory.logs.append(
        {
            "stage": memory.stage,
            "active_color": memory.active_color,
            "held_color": memory.held_color,
            "delivered_count": memory.delivered_count,
            "observation": {
                "visible_count": len(observation.detections),
                "note": observation.note,
                "colors": sorted({getattr(d, "color", "unknown") for d in observation.detections}),
            },
            "llm_decision": decision.__dict__,
            "verified": verified,
            "failed_attempts": dict(memory.failed_attempts),
        }
    )


# ---------------------------------------------------------------------------
# LEVEL 2 학생 TODO: vision-only action 구현
# ---------------------------------------------------------------------------
# Level 2는 go_to를 호출하면 안 됩니다. Camera observation, set_head,
# set_velocity, memory, recovery behavior로 navigate하세요.


async def visual_search(ctx: Any, target_color: str | None = None) -> bool:
    """Head scan과 짧은 body rotation으로 target color blob을 찾습니다."""
    scan_patterns = [
        (-0.95, -0.55, -0.2, 0.2, 0.55, 0.95),
        (-0.7, -0.25, 0.25, 0.7),
        (-0.9, 0.0, 0.9),
    ]
    for idx, yaws in enumerate(scan_patterns):
        detections = await scan_head(ctx, yaws=yaws, pitch=0.15)
        candidates = [d for d in detections if target_color is None or d.color == target_color]
        if candidates:
            best = max(candidates, key=lambda d: d.blob_area)
            head_yaw = max(-0.8, min(0.8, math.radians(getattr(best, "full_bearing_deg", best.angle_deg))))
            await set_head(ctx, yaw=head_yaw, pitch=0.15)
            return True
        turn = 0.35 if idx % 2 == 0 else -0.35
        await move_velocity(ctx, vx=0.08, wz=turn, duration_s=0.8)
    return False


async def visual_navigate_to_target(ctx: Any, target_color: str | None) -> bool:
    """정면 color blob angle/area feedback으로 closed-loop 접근합니다."""
    if target_color is None:
        return False

    arrival_area = 7000
    centered_tolerance_deg = 10.0
    last_seen_area = 0
    await set_head(ctx, yaw=0.0, pitch=0.15)
    await asyncio.sleep(0.2)

    for _step in range(16):
        raw_detections = await perceive(ctx)
        candidates = [
            ScannedDetection(
                color=d.color,
                angle_deg=d.angle_deg,
                blob_area=d.blob_area,
                centroid=d.centroid,
                bbox=d.bbox,
                head_yaw=0.0,
                head_pitch=0.15,
            )
            for d in raw_detections
            if d.color == target_color
        ]
        target = max(candidates, key=lambda d: d.blob_area, default=None)
        if target is None:
            await move_velocity(ctx, vx=0.10, wz=0.30, duration_s=0.6)
            continue

        angle = target.angle_deg
        area = target.blob_area
        last_seen_area = max(last_seen_area, area)
        if area >= arrival_area and abs(angle) <= 18.0:
            await move_velocity(ctx, vx=0.08, wz=0.0, duration_s=0.3)
            return True

        if abs(angle) > centered_tolerance_deg:
            # Positive image angle means target is to one side; use a short yaw correction,
            # then re-observe instead of committing to a long blind move.
            wz = -0.32 if angle > 0 else 0.32
            await move_velocity(ctx, vx=0.12, wz=wz, duration_s=0.55)
        else:
            speed = 0.35 if area < arrival_area * 0.55 else 0.20
            await move_velocity(ctx, vx=speed, wz=0.0, duration_s=0.65)

    return last_seen_area >= arrival_area * 0.80


async def recover_motion(ctx: Any, memory: AgentMemory, reason: str | None = None) -> dict[str, Any]:
    """반복 실패 횟수에 따라 후진, 회전, 재스캔 폭을 조절합니다."""
    worst_failure = max(memory.failed_attempts.values(), default=0)
    await cancel_action(ctx)
    await move_velocity(ctx, vx=-0.18, duration_s=0.8)
    turn = 0.35 if worst_failure % 2 == 0 else -0.35
    await move_velocity(ctx, vx=0.08, wz=turn, duration_s=1.0)
    detections = await scan_head(ctx, yaws=(-0.95, -0.35, 0.35, 0.95), pitch=0.15)
    target_color = memory.held_color or memory.active_color
    target_seen = any(d.color == target_color for d in detections) if target_color else bool(detections)
    return {
        "action": "recover",
        "reason": reason,
        "status": "recovered" if target_seen else "recovered_no_target_seen",
        "visible_count": len(detections),
        "target_seen": target_seen,
        "worst_failure": worst_failure,
    }


async def execute_decision(
    ctx: Any,
    decision: AgentDecision,
    observation: Observation,
    memory: AgentMemory,
) -> dict[str, Any]:
    """검증된 LLM decision 하나를 Level 2 robot action으로 변환합니다."""
    if decision.next_action in {"search_cube", "search_pad"}:
        target_color = decision.target_color or (memory.held_color if decision.next_action == "search_pad" else memory.active_color)
        found = await visual_search(ctx, target_color)
        return {"action": decision.next_action, "found": found, "target_color": target_color}

    if decision.next_action == "navigate_to_cube":
        reached = await visual_navigate_to_target(ctx, decision.target_color)
        return {"action": decision.next_action, "reached": reached}

    if decision.next_action == "navigate_to_pad":
        target_color = decision.target_color or memory.held_color
        reached = await visual_navigate_to_target(ctx, target_color)
        return {"action": decision.next_action, "reached": reached, "target_color": target_color}

    if decision.next_action == "pick_cube":
        if memory.stage != "ready_to_pick" or not memory.active_color:
            return {"action": "pick_cube", "status": "blocked_precondition", "reason": "not visually confirmed near selected cube"}
        result = await pick_nearest_cube(ctx)
        return {"action": "pick_cube", "result": result_summary(result)}

    if decision.next_action == "place_cube":
        if memory.stage != "ready_to_place" or not memory.held_color:
            return {"action": "place_cube", "status": "blocked_precondition", "reason": "not visually confirmed near matching pad"}
        result = await place_nearest_zone(ctx)
        return {"action": "place_cube", "result": result_summary(result)}

    if decision.next_action == "recover":
        return await recover_motion(ctx, memory, decision.recovery_strategy)

    if decision.next_action == "skip_target":
        return {"action": "skip_target", "status": "skipped"}

    return {"action": decision.next_action, "status": "unknown_action", "reason": "decision action was not executable"}


async def run_agent(
    ctx: Any,
    *,
    max_cycles: int = 80,
    completion: CompletionConfig | None = None,
) -> AgentMemory:
    """observe -> LLM decide -> deterministic execute -> verify -> memory loop."""
    memory = AgentMemory()
    last_result: dict[str, Any] | None = None
    tracker = CompletionTracker(completion) if completion is not None else None
    if completion is not None:
        memory.delivery_limit = completion.max_delivered_cubes

    for cycle in range(1, max_cycles + 1):
        print(f"\n[Level 2] Cycle {cycle}")
        if tracker is not None:
            first_cycle = tracker.started_at is None
            tracker.start_first_cycle()
            if first_cycle:
                tracker.print_start()
            reason = tracker.stop_reason(memory.delivered_count)
            if reason is not None:
                tracker.mark_ended(reason)
                print(f"Completion target reached before cycle action: {reason}.")
                break

        if max(memory.failed_attempts.values(), default=0) >= 8:
            last_result = await recover_motion(ctx, memory, "max_failure_guard")
            memory.logs.append({"guard_recovery": last_result, "failed_attempts": dict(memory.failed_attempts)})
            # Keep trying within the 10-minute cap, but avoid one failure key growing forever.
            memory.failed_attempts.clear()

        observation = await observe_world(ctx, memory)
        decision = await decide_next_action(TASK, observation, memory, last_result)
        print("LLM decision:", decision)

        if decision.next_action == "stop":
            update_memory(memory, observation, decision, {"ok": True, "stop_reason": decision.reason, "delivered_count": memory.delivered_count})
            break

        action_result = await execute_decision(ctx, decision, observation, memory)
        print("Action result:", action_result)
        verified = await verify_outcome(ctx, decision, action_result, memory)
        print(
            "Verified:",
            {
                "ok": verified.get("ok"),
                "target_seen": verified.get("target_seen"),
                "max_target_area": verified.get("max_target_area"),
                "sdk_status": verified.get("sdk_status"),
                "held_color": verified.get("held_color"),
                "delivered_count": verified.get("delivered_count"),
            },
        )
        update_memory(memory, observation, decision, verified)
        last_result = verified

        if tracker is not None:
            reason = tracker.stop_reason(memory.delivered_count)
            if reason is not None:
                tracker.mark_ended(reason)
                print(f"Completion target reached after cycle action: {reason}.")
                break

    if tracker is not None:
        tracker.print_summary(memory.delivered_count)
    return memory


async def run(ctx: Any) -> None:
    print(TASK)
    print("Level 2 autonomous-vision project starter 실행")
    memory = await run_agent(
        ctx,
        max_cycles=10_000,
        completion=CompletionConfig(level=2, max_elapsed_s=600),
    )
    print("\n실행 완료.")
    print(f"Delivered count: {memory.delivered_count}")
    print("Logs:")
    for item in memory.logs:
        print(item)



## Robot 연결

Prompt가 나오면 출력된 viewer URL을 Chrome에서 여세요.


In [ ]:
from menlo_runner.config import load_config
from menlo_runner.context import open_context

cfg = load_config()
ctx = await open_context(cfg, "project")
print("Viewer:", ctx.viewer_url)


## Agent 실행

현재 실험에 필요한 TODO를 충분히 구현한 뒤 실행하세요. 이 cell은 기본적으로 10분 scored simulation을 실행합니다.


In [ ]:
memory = await run_agent(
    ctx,
    max_cycles=10_000,
    completion=CompletionConfig(level=2, max_elapsed_s=600),
)
memory


In [ ]:
# 종료할 때 cleanup하세요.
# await ctx.close()
